# CODD PBL3 - Photo-input models (no manual damage grades)
These models use the 24 features that exist for a new photo: material metadata plus image-derived features. The six manual damage grade columns are **not** used, so this is the realistic setting for a user uploading a photo. Models are trained by `photo_only_comparison.py` (hyperparameters chosen on validation only, then refit on train + validation) and saved in `photo_models/`. The main 30-feature models are unchanged.

In [1]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)

import codd_models  # needed to load the Mixture of Experts
from codd_common import ANNOTATION_FEATURES, CLASS_NAMES, FEATURE_COLUMNS, load_grouped_splits

data = load_grouped_splits()
PHOTO_DIR = Path("photo_models")
cols = joblib.load(PHOTO_DIR / "photo_feature_columns.joblib")
assert len(cols) == 24 and not set(cols) & set(ANNOTATION_FEATURES)
assert cols == [c for c in FEATURE_COLUMNS if c not in ANNOTATION_FEATURES]
X_test, y_test = data.X_test[cols], data.y_test
print("Photo-input features:", len(cols))
print(cols)

Grouped splits: train (11465, 30) val (2410, 30) test (2644, 30)
Image_ID overlap (must all be 0): {'train-val': 0, 'train-test': 0, 'val-test': 0}
Features: 30 (24 numeric, 6 categorical)
Photo-input features: 24
['Material_Type', 'Material_Subtype', 'Source', 'Exposure_Type', 'pixel_obj_width', 'pixel_obj_height', 'relative_area', 'edge_density', 'fill_ratio', 'color_std', 'mean_brightness', 'std_brightness', 'contrast', 'texture_laplacian_var', 'dark_pixel_ratio', 'bright_pixel_ratio', 'mean_hue', 'mean_saturation', 'saturation_std', 'contour_area', 'contour_perimeter', 'circularity', 'solidity', 'extent']


In [2]:
files = {"SVM": "photo_svm", "CART": "photo_cart", "GAM": "photo_gam",
         "Random Forest": "photo_random_forest", "MoE": "photo_moe", "Stacking": "photo_stacking"}
models = {name: joblib.load(PHOTO_DIR / f"{f}.joblib") for name, f in files.items()}
rows, preds = [], {}
for name, model in models.items():
    p = model.predict(X_test); preds[name] = p
    rows.append({"Model": name,
                 "Accuracy": accuracy_score(y_test, p),
                 "Macro Precision": precision_score(y_test, p, average="macro", zero_division=0),
                 "Macro Recall": recall_score(y_test, p, average="macro", zero_division=0),
                 "Macro F1": f1_score(y_test, p, average="macro", zero_division=0)})
photo_results = pd.DataFrame(rows).sort_values("Macro F1", ascending=False).reset_index(drop=True)
photo_results.round(4)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,GAM,0.8491,0.6941,0.8813,0.7587
1,Stacking,0.8601,0.6914,0.8955,0.7555
2,CART,0.8222,0.6625,0.7547,0.6941
3,SVM,0.8404,0.6702,0.7171,0.6904
4,Random Forest,0.8529,0.7682,0.6462,0.6882
5,MoE,0.8011,0.6743,0.6747,0.6703


In [3]:
# Detailed test results for every photo-input model (same layout as the main notebooks)
from classification_reports import format_report

for name in ["SVM", "CART", "GAM", "Random Forest", "MoE", "Stacking"]:
    print(format_report(name, y_test, preds[name], "photo input, 24 features"))
    print("=" * 60)


SVM Test Results (photo input, 24 features)
-------------------------------------------
Accuracy : 0.8404
Precision (macro): 0.6702
Recall (macro)   : 0.7171
F1-score (macro) : 0.6904

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.31      0.38      0.34        32
        Good       0.70      0.85      0.76       586
        Fair       0.93      0.85      0.89      1788
        Poor       0.75      0.79      0.77       238

    accuracy                           0.84      2644
   macro avg       0.67      0.72      0.69      2644
weighted avg       0.85      0.84      0.84      2644

CART Test Results (photo input, 24 features)
--------------------------------------------
Accuracy : 0.8222
Precision (macro): 0.6625
Recall (macro)   : 0.7547
F1-score (macro) : 0.6941

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.32      0.66      0.43        32
        Good       0.71      0.74     

MoE Test Results (photo input, 24 features)
-------------------------------------------
Accuracy : 0.8011
Precision (macro): 0.6743
Recall (macro)   : 0.6747
F1-score (macro) : 0.6703

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.48      0.38      0.42        32
        Good       0.65      0.81      0.72       586
        Fair       0.89      0.82      0.86      1788
        Poor       0.68      0.70      0.69       238

    accuracy                           0.80      2644
   macro avg       0.67      0.67      0.67      2644
weighted avg       0.81      0.80      0.80      2644

Stacking Test Results (photo input, 24 features)
------------------------------------------------
Accuracy : 0.8601
Precision (macro): 0.6914
Recall (macro)   : 0.8955
F1-score (macro) : 0.7555

Classification Report:
              precision    recall  f1-score   support

   Excellent       0.38      0.94      0.54        32
        Good       0.76      0

In [4]:
# With vs without manual damage grades (test split)
both = pd.read_csv("model_comparison_results/with_vs_without_damage_grades.csv")
both.round(4)

,Model,Accuracy (30 features),Macro F1 (30 features),"Accuracy (photo-only, 24)","Macro F1 (photo-only, 24)"
0,SVM,0.9992,0.9953,0.8404,0.6904
1,CART,0.9970,0.9851,0.8222,0.6941
2,GAM,0.9962,0.9782,0.8491,0.7587
3,Random Forest,0.9879,0.9733,0.8529,0.6882
4,MoE,0.9508,0.9126,0.8011,0.6703
5,Stacking,0.9992,0.9957,0.8601,0.7555


In [5]:
best = photo_results.iloc[0]["Model"]
print("Best photo-input model by macro F1:", best)
print(classification_report(y_test, preds[best], labels=CLASS_NAMES, zero_division=0))
cm = confusion_matrix(y_test, preds[best], labels=CLASS_NAMES)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f"{best} (photo input) - Test Confusion Matrix")
plt.xlabel("Predicted Class"); plt.ylabel("Actual Class")
plt.tight_layout(); plt.show()

Best photo-input model by macro F1: GAM
              precision    recall  f1-score   support

   Excellent       0.45      0.88      0.60        32
        Good       0.75      0.88      0.81       586
        Fair       0.97      0.82      0.89      1788
        Poor       0.60      0.94      0.73       238

    accuracy                           0.85      2644
   macro avg       0.69      0.88      0.76      2644
weighted avg       0.88      0.85      0.86      2644



In [6]:
# Example: predict the condition of new items from photo-derived features only
sample = X_test.head(5)
pd.DataFrame({"Predicted": models["Stacking"].predict(sample), "Actual": y_test.head(5).values})

,Predicted,Actual
0,Good,Fair
1,Fair,Fair
2,Fair,Fair
3,Good,Excellent
4,Fair,Fair
